In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

In [6]:
def add_features(df):
    df = df.copy()
    df['prev_success']        = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted']     = (df['previous'] == 0).astype(int)
    df['pdays_clean']         = df['pdays'].apply(lambda x: 999 if x == -1 else x)
    df['prev_contacts_log']   = np.log1p(df['previous'])
    df['duration_log']        = np.log1p(df['duration'])
    df['log_balance']         = np.log1p(df['balance'].clip(lower=0))
    df['is_debt']             = (df['balance'] < 0).astype(int)
    df['log_campaign']        = np.log1p(df['campaign'])
    df['month_sin']           = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']           = np.cos(2 * np.pi * df['month'] / 12)
    df['long_call']           = (df['duration'] > 300).astype(int)
    df['long_call_x_success'] = df['long_call'] * df['prev_success']
    df['duration_x_prev']     = df['duration_log'] * df['prev_contacts_log']
    df['duration_x_success']  = df['duration_log'] * df['prev_success']
    df['pdays_x_prev']        = (1 / (df['pdays_clean'] + 1)) * df['prev_contacts_log']
    df['balance_x_debt']      = df['log_balance'] * (1 - df['is_debt'])
    df['duration_per_contact']= df['duration_log'] / (df['log_campaign'] + 1)
    df['age_young']           = (df['age'] < 30).astype(int)
    df['age_senior']          = (df['age'] >= 55).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)
print('TRAIN_DATA shape:', TRAIN_DATA.shape)

TRAIN_DATA shape: (29839, 35)


In [7]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import StratifiedKFold

cat_cols = ['job','marital_status','education','default_loan',
            'housing_loan','personal_loan','contact_type','poutcome']
num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

ENCODER = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1
                        ).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index)
    return pd.concat([cat_enc, df[num_cols].copy()], axis=1)

def target_encode_cv(X_tr, y_tr, X_te, cols, n_splits=5):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_tr_enc, X_te_enc = X_tr.copy(), X_te.copy()
    global_mean = y_tr.mean()
    for col in cols:
        oof, te_vals = np.full(len(X_tr), global_mean), np.zeros(len(X_te))
        for tri, vali in skf.split(X_tr, y_tr):
            means = y_tr.iloc[tri].groupby(X_tr[col].iloc[tri]).mean()
            oof[vali] = X_tr[col].iloc[vali].map(means).fillna(global_mean).values
            te_vals  += X_te[col].reset_index(drop=True).map(means).fillna(global_mean).values / n_splits
        X_tr_enc[col+'_te'] = oof
        X_te_enc[col+'_te'] = te_vals
    return X_tr_enc, X_te_enc

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values
y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)
print('X_train_te shape:', X_train_te.shape)

X_train_te shape: (29839, 43)


In [8]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

scale_pos = (y_train == 0).sum() / (y_train == 1).sum()
X_arr    = X_train_te.values
X_te_arr = X_test_te.values

best_xgb = {
    'n_estimators'     : 1398,
    'learning_rate'    : 0.014688107933307866,
    'max_depth'        : 9,
    'min_child_weight' : 11,
    'subsample'        : 0.9081675632335575,
    'colsample_bytree' : 0.9154397069234146,
    'colsample_bylevel': 0.7859644524805499,
    'reg_alpha'        : 14.449527198623775,
    'reg_lambda'       : 0.0013131988451547652,
    'gamma'            : 1.3271607503475447,
    'scale_pos_weight' : scale_pos,
    'eval_metric'      : 'logloss',
    'n_jobs'           : -1,
}

best_lgbm = {
    'boosting_type'    : 'gbdt',
    'n_estimators'     : 1544,
    'learning_rate'    : 0.009776236382241175,
    'max_depth'        : 12,
    'num_leaves'       : 189,
    'min_child_samples': 79,
    'subsample'        : 0.6758166608075022,
    'colsample_bytree' : 0.6287663520283125,
    'reg_alpha'        : 14.059097249147474,
    'reg_lambda'       : 0.5598467566307593,
    'class_weight'     : 'balanced',
    'n_jobs'           : -1,
    'verbose'          : -1,
}

best_cat = {
    'iterations'         : 639,
    'learning_rate'      : 0.013197977619438363,
    'depth'              : 10,
    'l2_leaf_reg'        : 9.52653108639879,
    'bagging_temperature': 1.422724713850696,
    'random_strength'    : 0.14507304848953856,
    'border_count'       : 167,
    'auto_class_weights' : 'Balanced',
    'eval_metric'        : 'Logloss',
    'verbose'            : 0,
}

print(f'scale_pos_weight: {scale_pos:.4f}')
print('Params loaded ✓')

scale_pos_weight: 7.5597
Params loaded ✓


In [9]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, roc_curve

# 5 seeds instead of 3 — more variance reduction, still fast since no Optuna
SEEDS    = [42, 2, 125, 17, 99]
N_SPLITS = 10
model_names = ['XGB', 'LGBM', 'CAT']

oof_preds  = {n: np.zeros(len(y_train))   for n in model_names}
test_preds = {n: np.zeros(len(X_te_arr))  for n in model_names}

for seed in SEEDS:
    print(f'\n--- Seed {seed} ---')
    skf      = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof_seed = {n: np.zeros(len(y_train))  for n in model_names}
    te_seed  = {n: np.zeros(len(X_te_arr)) for n in model_names}

    for fold, (tri, vali) in enumerate(skf.split(X_arr, y_train)):
        X_tr, X_val = X_arr[tri], X_arr[vali]
        y_tr        = y_train[tri]

        m = XGBClassifier(**best_xgb, random_state=seed)
        m.fit(X_tr, y_tr)
        oof_seed['XGB'][vali] = m.predict_proba(X_val)[:, 1]
        te_seed['XGB']       += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        m = LGBMClassifier(**best_lgbm, random_state=seed)
        m.fit(X_tr, y_tr)
        oof_seed['LGBM'][vali] = m.predict_proba(X_val)[:, 1]
        te_seed['LGBM']       += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        m = CatBoostClassifier(**best_cat, random_seed=seed)
        m.fit(X_tr, y_tr)
        oof_seed['CAT'][vali] = m.predict_proba(X_val)[:, 1]
        te_seed['CAT']       += m.predict_proba(X_te_arr)[:, 1] / N_SPLITS

        print(f'  Fold {fold+1}/{N_SPLITS} done')

    for n in model_names:
        oof_preds[n]  += oof_seed[n]  / len(SEEDS)
        test_preds[n] += te_seed[n]   / len(SEEDS)

print('\nOOF BA per model:')
for n in model_names:
    fpr, tpr, ths = roc_curve(y_train, oof_preds[n])
    t  = float(ths[np.argmax(tpr - fpr)])
    ba = balanced_accuracy_score(y_train, (oof_preds[n] >= t).astype(int))
    print(f'  {n}: BA={ba:.5f}  threshold={t:.4f}')


--- Seed 42 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

--- Seed 2 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

--- Seed 125 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

--- Seed 17 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

--- Seed 99 ---
  Fold 1/10 done
  Fold 2/10 done
  Fold 3/10 done
  Fold 4/10 done
  Fold 5/10 done
  Fold 6/10 done
  Fold 7/10 done
  Fold 8/10 done
  Fold 9/10 done
  Fold 10/10 done

OOF BA per model:
  XGB: BA=0.87512  threshold=0.3605
  LGB

In [10]:
from scipy.stats import rankdata

def rank_norm(arr):
    return rankdata(arr) / len(arr)

oof_rn  = np.column_stack([rank_norm(oof_preds[n])  for n in model_names])
test_rn = np.column_stack([rank_norm(test_preds[n]) for n in model_names])

oof_blend  = oof_rn.mean(axis=1)
test_blend = test_rn.mean(axis=1)

fpr, tpr, ths = roc_curve(y_train, oof_blend)
best_t  = float(ths[np.argmax(tpr - fpr)])
best_ba = balanced_accuracy_score(y_train, (oof_blend >= best_t).astype(int))

print(f'Blended OOF BA : {best_ba:.5f}')
print(f'Threshold      : {best_t:.4f}')
print(f'Expected LB    : ~{best_ba + 0.008:.5f}')

print('\nThreshold scan:')
for t in np.arange(max(0.01, best_t-0.05), min(0.99, best_t+0.06), 0.005):
    preds = (oof_blend >= t).astype(int)
    ba    = balanced_accuracy_score(y_train, preds)
    mark  = ' <- best' if abs(t - best_t) < 0.003 else ''
    print(f'  {t:.3f} | {preds.sum():6d} | {ba:.5f}{mark}')

Blended OOF BA : 0.87633
Threshold      : 0.7451
Expected LB    : ~0.88433

Threshold scan:
  0.695 |   9046 | 0.86609
  0.700 |   8902 | 0.86785
  0.705 |   8772 | 0.86853
  0.710 |   8623 | 0.87022
  0.715 |   8484 | 0.87074
  0.720 |   8314 | 0.87169
  0.725 |   8155 | 0.87292
  0.730 |   8002 | 0.87388
  0.735 |   7858 | 0.87450
  0.740 |   7715 | 0.87494
  0.745 |   7556 | 0.87633 <- best
  0.750 |   7412 | 0.87549
  0.755 |   7287 | 0.87575
  0.760 |   7128 | 0.87503
  0.765 |   6983 | 0.87389
  0.770 |   6841 | 0.87366
  0.775 |   6678 | 0.87301
  0.780 |   6552 | 0.87281
  0.785 |   6401 | 0.87242
  0.790 |   6233 | 0.87074
  0.795 |   6088 | 0.86894
  0.800 |   5942 | 0.86749


In [11]:
test_classes = (test_blend >= best_t).astype(int)
print(f'Distribution — 0: {(test_classes==0).sum()}, 1: {test_classes.sum()} '
      f'(pos-rate {test_classes.mean()*100:.1f}%)')

submission = pd.DataFrame({'id': TEST_DATA.index, 'subscription': test_classes})
submission.to_csv('submission_p1.csv', index=False)
print('Saved submission_v7.csv ✓')

Distribution — 0: 14834, 1: 5059 (pos-rate 25.4%)
Saved submission_v7.csv ✓
